# Operaciones de aprendizaje automático
## Maestría en Inteligencia Artificial Aplicada

### TC5061 | Proyecto MiBici: propuesta y base reproducible | Colaborativo

#### **Equipo:** 7
#### **Nombre:** Daniel Sánchez Hernández
#### **Matrícula:** A01840324

## Objetivo
Notebook de **exploración** del proyecto MiBici: carga y descripción de los datos abiertos de viajes y estaciones, análisis exploratorio de datos (EDA) y evidencia para las decisiones de limpieza de datos que acordará el equipo.

La pregunta que me corresponde es la **P1 (regresión)**: ¿cuántos viajes se iniciarán en una estación o zona en un periodo futuro?

La lógica que se reutilice (descarga, normalización y *features*) se trasladará al módulo del proyecto; en este notebook se desarrolla y se documenta la evidencia que la justifica.

## Dataset
**Datos abiertos de MiBici**, el sistema de bicicleta pública del Área Metropolitana de Guadalajara. El portal publica:
- **Viajes por mes** (`datos_abiertos_AAAA_MM.csv`) desde diciembre de 2014. Cada registro es un viaje con identificador, datos del usuario, fecha y hora de inicio y fin, y estaciones de origen y destino.
- **Nomenclatura de estaciones** (`nomenclatura_AAAA_MM.csv`) con identificador, nombre, latitud, longitud y capacidad de cada estación.

- Fuente: [Datos abiertos de MiBici](https://mibici.net/es/datos-abiertos/)
- Rango analizado: **enero de 2020 a septiembre de 2026**, acordado por el equipo para homologar el EDA. Puede recortarse después con base en los hallazgos. El portal ya no publica 2023-04 ni 2023-05 (ver la decisión de EDA en 1.1).

## Nota de reproducibilidad
El notebook se ejecuta desde la carpeta `notebooks/` con el entorno descrito en el README. La primera ejecución descarga los archivos a `data/raw/`, que no se versiona en Git; las siguientes ejecuciones reutilizan los archivos que ya existen.

In [1]:
# Importación de librerías y registro de versiones (evidencia de reproducibilidad)
import codecs
from pathlib import Path
import re
import sys
import time
from urllib.parse import urljoin

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow
import requests
import seaborn as sns

print("Python      :", sys.version.split()[0])
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("pyarrow     :", pyarrow.__version__)
print("requests    :", requests.__version__)
print("matplotlib  :", matplotlib.__version__)
print("seaborn     :", sns.__version__)

Python      : 3.11.17
pandas      : 3.0.5
numpy       : 1.26.4
pyarrow     : 25.0.1
requests    : 2.34.2
matplotlib  : 3.11.2
seaborn     : 0.13.2


## Configuración global

Todos los valores que se usan en más de una celda se definen una sola vez aquí:
- **Rutas:** carpetas de datos crudos de viajes y de estaciones, dentro de `data/raw/`.
- **Fuente y rango:** URL del portal, meses inicial y final (inclusivos), meses que el portal ya no publica y encabezados HTTP de las peticiones.
- **Random seed:** para que cualquier muestreo sea reproducible.
- **Opciones de visualización:** pandas muestra todas las columnas y filas suficientes para las tablas de inventario.
- **Colores:** paleta Okabe-Ito, apta para daltonismo, referida por nombre.

In [2]:
# Rutas del proyecto (el notebook se ejecuta desde notebooks/)
RUTA_PROYECTO = Path.cwd().parent
RUTA_VIAJES_CRUDOS = RUTA_PROYECTO / "data" / "raw" / "viajes"
RUTA_ESTACIONES_CRUDAS = RUTA_PROYECTO / "data" / "raw" / "estaciones"

# Fuente de los datos y rango de meses a analizar
URL_PORTAL = "https://mibici.net/es/datos-abiertos/"
MES_INICIO = "2020-01"
MES_FIN = "2026-09"

# Agregado en una revisión posterior: el portal dejó de publicar estos meses después de la
# primera descarga (13-oct-2026). Solo estos huecos se aceptan; cualquier otro detiene el notebook.
MESES_SIN_PUBLICAR = ["2023-04", "2023-05"]

# El portal rechaza (error 406) las peticiones con el User-Agent por defecto de requests
ENCABEZADOS_HTTP = {"User-Agent": "Mozilla/5.0 (proyecto academico MLOps TC5061)",
                    "Accept": "text/html,text/csv,*/*",
}

SEMILLA = 42

# Opciones de visualización de pandas
pd.set_option("display.max_columns", None)   # mostrar todas las columnas
pd.set_option("display.max_rows", 120)       # suficiente para el inventario mensual
pd.set_option("display.width", 200)          # ancho suficiente para tablas amplias
pd.set_option("display.max_colwidth", None)  # no recortar textos largos

# Paleta Okabe-Ito (apta para daltonismo)
AZUL = "#0072B2"
NARANJA = "#E69F00"
AMARILLO = "#F0E442"
VERDE = "#009E73"

RUTA_VIAJES_CRUDOS.mkdir(parents=True, exist_ok=True)
RUTA_ESTACIONES_CRUDAS.mkdir(parents=True, exist_ok=True)

## Paso 1: Carga y descripción del dataset

En este paso se descargan los archivos del portal y se revisa su estructura antes de unirlos. Las instrucciones de la actividad advierten que los nombres de columnas y la codificación de caracteres pueden variar entre años, por lo que primero se hace un **inventario de cada archivo**. Las diferencias que aparezcan se documentan como hallazgos y se resuelven con evidencia, no con supuestos.

### 1.1 Descarga de los archivos

Las URLs del portal no se pueden construir a partir del año y el mes: cada archivo vive en una carpeta con un identificador distinto (`/site/assets/files/{id}/datos_abiertos_AAAA_MM.csv`). Por eso los enlaces se leen directamente de la página del portal y se filtran por el rango de meses.

Si falta en el portal algún mes del rango que no esté registrado en `MESES_SIN_PUBLICAR`, el notebook se detiene con un error. Así un cambio nuevo del portal no pasa desapercibido en el resto del análisis.

Cada archivo se descarga una sola vez: si ya existe en `data/raw/`, se omite.

In [3]:
def leer_portal(url_portal):
    """Regresa el HTML de la página de datos abiertos."""
    respuesta = requests.get(url_portal, headers=ENCABEZADOS_HTTP, timeout=30)
    respuesta.raise_for_status()
    return respuesta.text


def extraer_enlaces(html, url_base, prefijo):
    """Regresa un DataFrame con archivo, periodo y URL de los CSV cuyo nombre inicia con el prefijo."""
    patron = rf"(?P<archivo>{prefijo}_(?P<anio>\d{{4}})_(?P<mes>\d{{2}})\.csv)"
    enlaces = {}
    for href in re.findall(r"href=['\"]([^'\"]+)['\"]", html):
        coincidencia = re.search(patron, href)
        if coincidencia:
            enlaces[coincidencia["archivo"]] = (pd.Period(f"{coincidencia['anio']}-{coincidencia['mes']}", freq="M"),
                                                urljoin(url_base, href))
    tabla = pd.DataFrame([(archivo, periodo, url) for archivo, (periodo, url) in enlaces.items()],
                         columns=["archivo", "periodo", "url"])
    return tabla.sort_values("periodo").reset_index(drop=True)


def descargar_archivo(url, destino):
    """Descarga el archivo si no existe; regresa True si se descargó en esta ejecución."""
    if destino.exists():
        return False
    respuesta = requests.get(url, headers=ENCABEZADOS_HTTP, timeout=60)
    respuesta.raise_for_status()
    destino.write_bytes(respuesta.content)
    return True

In [4]:
# Enlaces publicados en el portal
html_portal = leer_portal(URL_PORTAL)
enlaces_viajes = extraer_enlaces(html_portal, URL_PORTAL, "datos_abiertos")
enlaces_estaciones = extraer_enlaces(html_portal, URL_PORTAL, "nomenclatura")
if enlaces_viajes.empty or enlaces_estaciones.empty:
    raise ValueError("No se encontraron enlaces en el portal; revisar si cambió la página o la URL")

print("Archivos de viajes publicados:", len(enlaces_viajes),
      f"({enlaces_viajes['periodo'].min()} a {enlaces_viajes['periodo'].max()})")
print("Archivos de nomenclatura publicados:", len(enlaces_estaciones))

# Meses del rango de análisis y meses que el portal no publica
meses_esperados = pd.period_range(MES_INICIO, MES_FIN, freq="M")
enlaces_viajes = enlaces_viajes[enlaces_viajes["periodo"].isin(meses_esperados)].reset_index(drop=True)
meses_faltantes = sorted(set(meses_esperados) - set(enlaces_viajes["periodo"]))

print(f"\nMeses en el rango {MES_INICIO} a {MES_FIN}:", len(meses_esperados))
print("Archivos de viajes en el rango:", len(enlaces_viajes))
print("Meses sin archivo en el portal:", [str(mes) for mes in meses_faltantes] or "ninguno")

# Solo se aceptan los huecos ya conocidos y documentados
meses_inesperados = [str(mes) for mes in meses_faltantes if str(mes) not in MESES_SIN_PUBLICAR]
if meses_inesperados:
    raise ValueError(f"Faltan meses no registrados en MESES_SIN_PUBLICAR: {meses_inesperados}")

Archivos de viajes publicados: 138 (2014-12 a 2026-09)
Archivos de nomenclatura publicados: 1

Meses en el rango 2020-01 a 2026-09: 81
Archivos de viajes en el rango: 79
Meses sin archivo en el portal: ['2023-04', '2023-05']


In [5]:
# Descarga de los viajes del rango y de la nomenclatura más reciente
nomenclatura_reciente = enlaces_estaciones.iloc[[-1]]
descargas = [(fila.url, RUTA_VIAJES_CRUDOS / fila.archivo) for fila in enlaces_viajes.itertuples()]
descargas += [(fila.url, RUTA_ESTACIONES_CRUDAS / fila.archivo) for fila in nomenclatura_reciente.itertuples()]

descargados = 0
for url, destino in descargas:
    if descargar_archivo(url, destino):
        descargados += 1
        time.sleep(1)  # pausa entre descargas para no saturar el portal

tamano_mb = sum(destino.stat().st_size for _, destino in descargas) / 1024**2
print("Archivos descargados en esta ejecución:", descargados)
print("Archivos que ya existían:", len(descargas) - descargados)
print(f"Tamaño total en disco: {tamano_mb:.1f} MB")
print("Nomenclatura utilizada:", nomenclatura_reciente["archivo"].iloc[0])

Archivos descargados en esta ejecución: 0
Archivos que ya existían: 80
Tamaño total en disco: 2097.6 MB
Nomenclatura utilizada: nomenclatura_2026_09.csv


**Observaciones de la descarga:**
- **Archivos publicados:** el portal publica 138 archivos mensuales de viajes (2014-12 a 2026-09) y una sola nomenclatura de estaciones (`nomenclatura_2026_09.csv`).
- **Cobertura del rango:** se publican 79 de los 81 meses de 2020-01 a 2026-09; faltan 2023-04 y 2023-05.
- **Meses retirados del portal:** en la primera ejecución (13-oct-2026) el portal publicaba 140 archivos, incluidos 2023-04 y 2023-05. Al día siguiente ya no aparecían; se buscó cualquier mención de esos meses en el HTML del portal, sin resultados, y se confirmó desde dos equipos. La fuente cambia sin aviso, por lo que el notebook compara los meses faltantes contra `MESES_SIN_PUBLICAR` y se detiene ante cualquier hueco nuevo.
- **Volumen descargado:** 80 archivos (79 de viajes y la nomenclatura), 2097.6 MB en total.
- **Acceso al portal:** el portal rechazó las primeras peticiones con un error 406 (*Not Acceptable*) por el User-Agent que `requests` envía por defecto. Se resolvió enviando encabezados HTTP explícitos, definidos en la configuración global; el script de ingesta debe conservar este ajuste.

**Decisión de EDA: meses retirados del portal (pendiente de acordar con el equipo)**

| Opción | Ventaja | Desventaja |
|---|---|---|
| **A. Trabajar con los 79 meses publicados y documentar el hueco** | Totalmente reproducible desde el portal | Las series mensuales de la P1 tienen un hueco de dos meses |
| B. Usar copias de 2023-04 y 2023-05 descargadas el 13-oct-2026, registradas en el manifiesto con su SHA-256 | Datos completos | Ya no se pueden reconstruir desde la fuente oficial |
| C. Esperar a que el portal los vuelva a publicar | Datos oficiales y completos | No hay certeza de que ocurra |

Se trabaja bajo la **opción A**. El tratamiento del hueco en las series agregadas se definirá en la limpieza de datos.

### 1.2 Inventario de los archivos de viajes

Antes de unir los archivos se revisa, para cada uno:
- **Codificación de caracteres:** se intenta leer como UTF-8 (detectando si trae BOM) y, si falla, se usa latin-1. UTF-8 tiene reglas estrictas, así que un archivo en latin-1 con acentos o "ñ" no pasa como UTF-8; latin-1 decodifica cualquier byte, por eso es el último recurso.
- **Registros:** número de líneas de datos, sin contar el encabezado.
- **Columnas:** nombres exactos tal como vienen en el archivo.
- **Formatos de fecha:** sobre una muestra de 2000 filas, cada valor con hora se convierte a su "forma", reemplazando cada dígito por `9` (por ejemplo, `2020-01-01 00:03:24` se convierte en `9999-99-99 99:99:99`). Así se detecta cualquier formato distinto sin suponer de antemano cuáles existen. Se investigó la técnica: es el perfilado de patrones (*pattern profiling*), una práctica estándar de data profiling para identificar los formatos presentes en una columna.

El inventario solo registra nombres de columnas y formatos; no muestra valores de las columnas con datos de usuarios.

In [6]:
def detectar_codificacion(contenido):
    """Regresa la codificación con la que se pueden decodificar los bytes del archivo."""
    if contenido.startswith(codecs.BOM_UTF8):
        return "utf-8-sig"
    try:
        contenido.decode("utf-8")
        return "utf-8"
    except UnicodeDecodeError:
        return "latin-1"  # decodifica cualquier byte; se usa como último recurso


def formas_de_fecha(muestra):
    """Regresa las formas (dígitos como 9) de los valores con hora en la muestra."""
    formas = set()
    for columna in muestra.columns:
        valores = muestra[columna].dropna()
        if valores.str.contains(":").any():
            formas.update(valores.str.replace(r"\d", "9", regex=True).unique())
    return sorted(formas)


def inventariar_archivo(ruta, filas_muestra=2000):
    """Regresa codificación, registros, columnas y formatos de fecha de un CSV."""
    contenido = ruta.read_bytes()
    codificacion = detectar_codificacion(contenido)
    lineas = contenido.count(b"\n") + (0 if contenido.endswith(b"\n") else 1)
    muestra = pd.read_csv(ruta, encoding=codificacion, nrows=filas_muestra, dtype=str)
    return {"archivo": ruta.name,
            "tamano_mb": round(len(contenido) / 1024**2, 1),
            "codificacion": codificacion,
            "registros": lineas - 1,
            "n_columnas": muestra.shape[1],
            "columnas": " | ".join(muestra.columns),
            "formatos_fecha": " | ".join(formas_de_fecha(muestra)),
    }

In [7]:
# Inventario de cada archivo de viajes del rango
inventario = pd.DataFrame([inventariar_archivo(RUTA_VIAJES_CRUDOS / archivo)
                           for archivo in enlaces_viajes["archivo"]])
inventario.insert(1, "periodo", enlaces_viajes["periodo"].astype(str))

print("Archivos inventariados:", len(inventario))
print("Registros totales:", inventario["registros"].sum())
print(f"Tamaño total: {inventario['tamano_mb'].sum():.1f} MB")
inventario[["archivo", "periodo", "tamano_mb", "codificacion", "registros", "n_columnas"]]

Archivos inventariados: 79
Registros totales: 26389162
Tamaño total: 2097.6 MB


,archivo,periodo,tamano_mb,codificacion,registros,n_columnas
0,datos_abiertos_2020_01.csv,2020-01,24.7,utf-8,360276,8
1,datos_abiertos_2020_02.csv,2020-02,25.2,utf-8,367864,8
2,datos_abiertos_2020_03.csv,2020-03,20.3,utf-8,296759,8
3,datos_abiertos_2020_04.csv,2020-04,8.9,utf-8,129276,8
4,datos_abiertos_2020_05.csv,2020-05,11.0,utf-8,160437,8
5,datos_abiertos_2020_06.csv,2020-06,13.4,utf-8,196071,8
6,datos_abiertos_2020_07.csv,2020-07,14.9,utf-8,217858,8
7,datos_abiertos_2020_08.csv,2020-08,15.8,utf-8,230399,8
8,datos_abiertos_2020_09.csv,2020-09,15.6,utf-8,228383,8
9,datos_abiertos_2020_10.csv,2020-10,16.8,utf-8,244543,8


In [8]:
# Esquemas distintos: archivos que comparten exactamente los mismos nombres de columnas
esquemas = (inventario.groupby("columnas")
            .agg(archivos=("archivo", "count"),
                 primer_periodo=("periodo", "min"),
                 ultimo_periodo=("periodo", "max"),
                 periodos=("periodo", lambda periodos: ", ".join(periodos)))
            .sort_values("archivos", ascending=False)
            .reset_index()
)
print("Esquemas de columnas distintos:", len(esquemas))
esquemas

Esquemas de columnas distintos: 2


,columnas,archivos,primer_periodo,ultimo_periodo,periodos
0,Viaje_Id | Usuario_Id | Genero | Año_de_nacimiento | Inicio_del_viaje | Fin_del_viaje | Origen_Id | Destino_Id,77,2020-01,2026-09,"2020-01, 2020-02, 2020-03, 2020-04, 2020-05, 2020-06, 2020-07, 2020-08, 2020-09, 2020-10, 2020-11, 2020-12, 2021-01, 2021-02, 2021-03, 2021-04, 2021-05, 2021-06, 2021-07, 2021-08, 2021-09, 2021-10, 2021-11, 2021-12, 2022-01, 2022-02, 2022-03, 2022-04, 2022-05, 2022-06, 2022-07, 2022-08, 2022-09, 2022-10, 2022-11, 2022-12, 2023-01, 2023-02, 2023-03, 2023-06, 2023-07, 2023-08, 2023-09, 2023-10, 2023-11, 2023-12, 2024-01, 2024-03, 2024-04, 2024-06, 2024-07, 2024-08, 2024-09, 2024-10, 2024-11, 2024-12, 2025-01, 2025-02, 2025-03, 2025-04, 2025-05, 2025-06, 2025-07, 2025-08, 2025-09, 2025-10, 2025-11, 2025-12, 2026-01, 2026-02, 2026-03, 2026-04, 2026-05, 2026-06, 2026-07, 2026-08, 2026-09"
1,Viaje_Id | Usuario_Id | Genero | A}äe_nacimiento | Inicio_del_viaje | Fin_del_viaje | Origen_Id | Destino_Id,2,2024-02,2024-05,"2024-02, 2024-05"


In [9]:
# Codificación de caracteres por archivo: conteo y porcentaje
conteo_codificacion = pd.concat([inventario["codificacion"].value_counts(),
                                 inventario["codificacion"].value_counts(normalize=True).mul(100).round(2)],
                                axis=1, keys=["count", "percent"],
)
print(conteo_codificacion)
print("-" * 45)

# Archivos que no están en UTF-8 simple
print(inventario.loc[inventario["codificacion"] != "utf-8", ["archivo", "codificacion"]].to_string(index=False))

              count  percent
codificacion                
latin-1          43    54.43
utf-8            36    45.57
---------------------------------------------
                   archivo codificacion
datos_abiertos_2021_06.csv      latin-1
datos_abiertos_2021_07.csv      latin-1
datos_abiertos_2021_08.csv      latin-1
datos_abiertos_2021_09.csv      latin-1
datos_abiertos_2021_10.csv      latin-1
datos_abiertos_2021_11.csv      latin-1
datos_abiertos_2021_12.csv      latin-1
datos_abiertos_2022_01.csv      latin-1
datos_abiertos_2022_02.csv      latin-1
datos_abiertos_2022_03.csv      latin-1
datos_abiertos_2022_04.csv      latin-1
datos_abiertos_2022_05.csv      latin-1
datos_abiertos_2022_06.csv      latin-1
datos_abiertos_2022_07.csv      latin-1
datos_abiertos_2022_08.csv      latin-1
datos_abiertos_2022_09.csv      latin-1
datos_abiertos_2022_10.csv      latin-1
datos_abiertos_2022_11.csv      latin-1
datos_abiertos_2023_01.csv      latin-1
datos_abiertos_2023_02.csv      latin-

In [10]:
# Formatos de fecha distintos y en qué archivos aparecen
formatos = (inventario.groupby("formatos_fecha")
            .agg(archivos=("archivo", "count"),
                 periodos=("periodo", lambda periodos: ", ".join(periodos)))
            .sort_values("archivos", ascending=False)
            .reset_index()
)
print("Combinaciones de formatos de fecha:", len(formatos))
formatos

Combinaciones de formatos de fecha: 3


,formatos_fecha,archivos,periodos
0,9999-99-99 99:99:99,76,"2020-01, 2020-02, 2020-03, 2020-04, 2020-05, 2020-06, 2020-07, 2020-08, 2020-09, 2020-10, 2020-11, 2020-12, 2021-01, 2021-02, 2021-03, 2021-04, 2021-05, 2021-06, 2021-07, 2021-08, 2021-09, 2021-10, 2021-11, 2021-12, 2022-01, 2022-02, 2022-03, 2022-04, 2022-05, 2022-06, 2022-07, 2022-08, 2022-09, 2022-10, 2022-11, 2022-12, 2023-01, 2023-02, 2023-03, 2023-06, 2023-07, 2023-08, 2023-09, 2023-10, 2023-12, 2024-01, 2024-02, 2024-03, 2024-04, 2024-05, 2024-06, 2024-07, 2024-08, 2024-11, 2024-12, 2025-01, 2025-02, 2025-03, 2025-04, 2025-05, 2025-06, 2025-07, 2025-08, 2025-09, 2025-10, 2025-11, 2025-12, 2026-01, 2026-02, 2026-03, 2026-04, 2026-05, 2026-06, 2026-07, 2026-08, 2026-09"
1,99/99/9999 99:99 | 99/99/9999 9:99,2,"2024-09, 2024-10"
2,99/99/9999 9:99,1,2023-11


**Observaciones del inventario de viajes:**
- **Volumen:** 26389162 registros en 79 archivos. El mes con menos viajes es 2020-04 (129276), que coincide con el inicio de la pandemia de COVID-19; el de más viajes es 2026-09 (490056).
- **Columnas:** todos los archivos tienen las mismas 8 columnas en el mismo orden, pero hay 2 esquemas. 77 archivos usan `Año_de_nacimiento` y 2 archivos (2024-02 y 2024-05) traen el nombre corrupto `A}äe_nacimiento`. Si se unieran sin corregirlo, pandas crearía dos columnas distintas y llenaría la otra con nulos; el nombre debe normalizarse antes de unir los archivos.
- **Codificación:** 43 archivos (54.43%) no son UTF-8 válido y se leen como latin-1; los otros 36 (45.57%) son UTF-8. Los archivos latin-1 se concentran entre 2021-06 y 2025-02, con excepciones en UTF-8 dentro de ese periodo (2022-12 y 2024-07) y dos casos aislados después (2025-07 y 2026-01). Como no sigue una regla por fecha, la codificación se detecta archivo por archivo en lugar de fijar una sola. La detección es confiable porque el encabezado de todos los archivos contiene al menos un carácter no ASCII (la "ñ" de `Año_de_nacimiento` o la "ä" del nombre corrupto).
- **Formatos de fecha:** 76 archivos usan `AAAA-MM-DD hh:mm:ss`. Tres archivos (2023-11, 2024-09 y 2024-10) usan día, mes y año separados por barras, sin segundos y con la hora en uno o dos dígitos (`9:99` y `99:99`). En 2023-11 solo apareció la hora de un dígito, probablemente porque la muestra cubre solo las primeras horas del mes. Por esto, al unir los archivos se usará un formato explícito por archivo en lugar de dejar que pandas lo infiera, ya que con formatos mezclados podría intercambiar día y mes sin avisar. La forma no distingue `DD/MM` de `MM/DD`; el orden se verificará comprobando que el mes de cada fecha coincida con el mes del archivo.
- **Limitación:** los formatos se detectaron en las primeras 2000 filas de cada archivo; la conversión completa de las fechas confirmará que no existen otros.

### 1.3 Inventario de la nomenclatura de estaciones

Se aplica la misma revisión al archivo de estaciones más reciente publicado en el portal. Este archivo no contiene datos de usuarios, por lo que sí se muestran sus primeras filas.

In [11]:
# Inventario y primeras filas de la nomenclatura de estaciones
ruta_nomenclatura = RUTA_ESTACIONES_CRUDAS / nomenclatura_reciente["archivo"].iloc[0]
inventario_estaciones = inventariar_archivo(ruta_nomenclatura)
for campo, valor in inventario_estaciones.items():
    print(f"{campo:<15}: {valor}")

estaciones = pd.read_csv(ruta_nomenclatura, encoding=inventario_estaciones["codificacion"])
display(estaciones.head())

archivo        : nomenclatura_2026_09.csv
tamano_mb      : 0.0
codificacion   : utf-8
registros      : 484
n_columnas     : 5
columnas       : id | name | latitude | longitude | dpcapacity
formatos_fecha : 


,id,name,latitude,longitude,dpcapacity
0,2,(GDL-001) C. Epigmenio Glez./ Av. 16 de Sept.,20.666378,-103.348820,15
1,3,(GDL-002) C. Colonias / Av. Niños héroes,20.667228,-103.366000,15
2,4,(GDL-003) C. Vidrio / Av. Chapultepec,20.667690,-103.368252,19
3,5,(GDL-004) C. Ghilardi /C. Miraflores,20.691847,-103.362549,19
4,6,(GDL-005) C. San Diego /Calzada Independencia,20.681158,-103.339363,11


**Observaciones de la nomenclatura de estaciones:**
- **Estructura:** 484 estaciones en UTF-8 con 5 columnas: `id`, `name`, `latitude`, `longitude` y `dpcapacity` (capacidad de anclajes?). A diferencia de los viajes, los nombres de columnas están en inglés; se conservan tal como vienen.
- **Código de estación:** `name` incluye un código entre paréntesis (por ejemplo, `(GDL-001)`) cuyo prefijo parece identificar el municipio. Puede servir para agrupar estaciones por zona en la pregunta de regresión.
- **Relación con los viajes:** el `id` no coincide con el número del código (el `id` 2 corresponde a `GDL-001`). La relación entre `Origen_Id`/`Destino_Id` y `id` se verificará al unir ambas tablas.
- **Vigencia:** es la nomenclatura vigente a 2026-09. Las estaciones dadas de baja antes de esa fecha podrían no aparecer, lo que también se revisará al unir las tablas.